# 183 — Complaints / Support / Transactions
Trains `models/183_model.pkl`. Checklist: plan.md §2. Heads: intent (Banking77), product (CFPB), risk/alert (creditcard fraud). 183's own JSON complaints are synthetic and used only as augmentation.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
while not (ROOT / "lib").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
from pathlib import Path

import yaml

cfg_path = ROOT / "config" / "run_config.yaml"
with open(cfg_path) as fh:
    RUN = yaml.safe_load(fh)
print(
    "run config:", {k: RUN[k] for k in ("active_models", "input_source", "eval_mode")}
)
TH = RUN.get("thresholds", {})

In [ ]:
import lib.artifacts as art
import lib.io_utils as io

io.ensure_dirs()

## 2.1 Data ingestion
Loads complaint/support/transaction families plus the real labeled corpora (CFPB, Banking77, creditcard fraud). CFPB is ~9 GB so it is streamed (first 100k rows).

In [ ]:
complaints = io.load_183_complaints()
print("183 source complaint/support/txn records:", len(complaints))
texts, labels = io.load_banking77()
print("Banking77 samples:", len(texts), "| intents:", len(set(labels)))
X_cc, y_cc = io.load_creditcard()
print("creditcard rows:", len(X_cc), f"| fraud rate: {y_cc.mean():.4f}")
cfpb = io.load_cfpb_sample(n=100_000)
print("CFPB sampled rows:", len(cfpb), "| products:", cfpb["Product"].nunique())

## 2.2 / 2.3 Feature engineering & labels
TF-IDF over complaint narratives; PCA transaction features for the risk head; supervised targets from Banking77 / CFPB / creditcard `Class`.

## 2.4 / 2.5 Modeling & evaluation

In [ ]:
import lib.model_183 as m183

model = m183.Model183().fit()
print("=== 183 metrics (TEST / held-out) ===")
for head, m in model.metrics.items():
    print(head, {k: round(v, 4) if isinstance(v, float) else v for k, v in m.items()})
print("--- TRAIN (fit) ---")
for head, m in model.train_metrics.items():
    print(head, {k: round(v, 4) if isinstance(v, float) else v for k, v in m.items()})

## 2.6 Output — serialize + smoke test

In [ ]:
from lib import schema

art.save_model(
    model,
    io.MODELS_DIR / "183_model.pkl",
    provenance={"model": "183", "metrics": model.metrics},
)
print("saved", io.MODELS_DIR / "183_model.pkl")
rec = next(
    (c for c in complaints if isinstance(c, dict) and c.get("fraud_details")),
    complaints[0],
)
payload = model.predict(rec, threshold=TH.get("alert", 0.7))
assert schema.is_valid(payload)
out_path = io.write_out(183, payload, "smoke", case_id=rec.get("complaint_id"))
print("smoke-test wrote:", out_path.name, "| validated: OK")